# Register Bronze Content Files

Registers GitHub files available in the execution-specific Bronze Lakehouse path in `bronze.content_files`.

The notebook discovers file metadata using Spark, prepares Bronze lineage and audit information, and returns the number of registered files for reconciliation.

## Input Parameters

Defines the execution step, project context, and audit information required to locate the Bronze files and register their metadata.

In [1]:
p_execution_step_id = 116
p_project_name = "DataOps_Control"
p_created_by = ""

StatementMeta(, bc404979-8603-4617-b8de-840c4822f110, 3, Finished, Available, Finished, False)

## Imports

Imports the Spark functions, storage level, and JSON utilities required for metadata preparation, DataFrame persistence, and notebook output.

In [2]:
import json
from datetime import datetime, timezone

from pyspark import StorageLevel
from pyspark.sql import functions as F

StatementMeta(, bc404979-8603-4617-b8de-840c4822f110, 4, Finished, Available, Finished, False)

## Build Bronze Context

Builds the execution-specific Bronze Lakehouse path for the selected GitHub project.

In [ ]:
def build_bronze_path(
    project_name,
    execution_step_id
):
    """
    Builds the execution-specific Bronze Lakehouse path
    for the selected GitHub project.
    """

    return (
        f"Files/bronze/github/"
        f"{project_name}/"
        f"execution_step_id={execution_step_id}"
    )

## Discover Bronze Files

Discovers all files under the execution-specific Bronze path using Spark recursive file lookup and returns the physical file path and size.

In [ ]:
def discover_bronze_files(
    lakehouse_path
):
    """
    Discovers Bronze files under the selected execution path
    using Spark recursive file lookup.
    """

    return (
        spark.read
        .format("binaryFile")
        .option("recursiveFileLookup", "true")
        .load(lakehouse_path)
        .select(
            F.col("path"),
            F.col("length").alias("file_size_bytes")
        )
    )

## Prepare Content File Metadata

Transforms discovered Bronze file information into the `bronze.content_files` structure, including source identification, file attributes, Bronze location, execution lineage, and audit metadata.

In [ ]:
def prepare_content_file_metadata(
    bronze_files,
    project_name,
    execution_step_id,
    created_at,
    created_by
):
    """
    Prepares discovered Bronze files for registration in
    bronze.content_files.
    """

    return (
        bronze_files

        # Physical path relative to the Lakehouse Files root.
        .withColumn(
            "bronze_path",
            F.concat(
                F.lit("Files/"),
                F.regexp_extract(
                    F.col("path"),
                    r"/Files/(.*)$",
                    1
                )
            )
        )

        # Original path within the ingested GitHub project.
        .withColumn(
            "source_path",
            F.regexp_extract(
                F.col("bronze_path"),
                rf"execution_step_id={execution_step_id}/(.*)$",
                1
            )
        )

        .withColumn(
            "file_name",
            F.element_at(
                F.split(F.col("source_path"), "/"),
                -1
            )
        )

        .withColumn(
            "file_type",
            F.when(
                F.instr(F.col("file_name"), ".") > 0,
                F.lower(
                    F.regexp_extract(
                        F.col("file_name"),
                        r"\.([^.]+)$",
                        1
                    )
                )
            )
        )

        .select(
            F.lit("GitHub").alias("source_system"),
            F.lit(project_name).alias("source_container"),
            "source_path",
            "file_name",
            "file_type",
            "file_size_bytes",
            "bronze_path",
            F.lit(execution_step_id)
                .cast("long")
                .alias("execution_step_id"),
            F.lit(created_at).alias("created_at"),
            F.lit(created_by).alias("created_by")
        )
    )

## Register Content Files

Persists the prepared Bronze content metadata, returns the number of registered files for reconciliation, and releases cached Spark resources after processing.

In [ ]:
def register_content_files(
    content_files
):
    """
    Registers prepared content file metadata in bronze.content_files
    and returns the number of registered files.
    """

    content_files.persist(StorageLevel.MEMORY_AND_DISK)

    try:
        files_copied_count = content_files.count()

        if files_copied_count > 0:
            (
                content_files.write
                .format("delta")
                .mode("append")
                .saveAsTable("bronze.content_files")
            )

        return files_copied_count

    finally:
        content_files.unpersist()

## Execute Registration Process

Orchestrates Bronze file discovery, metadata preparation, and registration for the selected GitHub project execution.

In [ ]:
created_at = datetime.now(timezone.utc)

lakehouse_path = build_bronze_path(
    project_name=p_project_name,
    execution_step_id=p_execution_step_id
)

bronze_files = discover_bronze_files(
    lakehouse_path=lakehouse_path
)

content_files = prepare_content_file_metadata(
    bronze_files=bronze_files,
    project_name=p_project_name,
    execution_step_id=p_execution_step_id,
    created_at=created_at,
    created_by=p_created_by
)

files_copied_count = register_content_files(
    content_files=content_files
)

## Output

Returns the number of registered Bronze files for downstream reconciliation.

In [7]:
notebookutils.notebook.exit(
    json.dumps({
        "files_copied_count": files_copied_count
    })
)

StatementMeta(, bc404979-8603-4617-b8de-840c4822f110, 9, Finished, Available, Finished, False)

ExitValue: {"files_copied_count": 32}